# RedAgent: Kev-Based Crypto Options Model Training (Google Colab)

This notebook fine-tunes **Jared Palmer's Kev** (`jaredpalmer/kev`) decision model using **LoRA** for crypto call options classification and signal generation.

### Recommended Colab Settings:
- **Runtime Type**: Python 3
- **Hardware Accelerator**: GPU (T4 for `kev-0.8b` or 4-bit `kev-4b`; A100 / L4 for native BF16 `kev-4b`)

### Pipeline Overview:
1. Mount Google Drive & Environment Setup
2. Clone & Install Kev Training Dependencies
3. Load Preprocessed Dataset (`workload.json`, `train.jsonl`, `val.jsonl`)
4. Execute LoRA Fine-Tuning
5. Evaluate Calibration & Brier Score
6. Export Trained Adapter to Google Drive for Local Mac Serving

In [ ]:
# Step 1: Check GPU Availability and VRAM
!nvidia-smi

In [ ]:
# Step 2: Mount Google Drive
from google.colab import drive
import os

drive.mount('/content/drive')

# Define paths in Google Drive
DRIVE_DIR = '/content/drive/MyDrive/redagent'
DATA_DIR = os.path.join(DRIVE_DIR, 'data')
OUTPUT_DIR = os.path.join(DRIVE_DIR, 'models/lora_adapter')

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Workspace Drive Directory: {DRIVE_DIR}")

In [ ]:
# Step 3: Clone Kev Repository and Install Dependencies
!rm -rf kev
!git clone https://github.com/jaredpalmer/kev.git
%cd kev

# Install packages
!pip install --upgrade pip
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install transformers peft accelerate bitsandbytes datasets scipy scikit-learn
!pip install -e .

In [ ]:
# Step 4: Verify Dataset Files and Workload Schema
import json

workload_path = os.path.join(DRIVE_DIR, 'workload.json')
train_path = os.path.join(DATA_DIR, 'train.jsonl')
val_path = os.path.join(DATA_DIR, 'val.jsonl')

assert os.path.exists(workload_path), f"workload.json missing at {workload_path}"
assert os.path.exists(train_path), f"train.jsonl missing at {train_path}"
assert os.path.exists(val_path), f"val.jsonl missing at {val_path}"

with open(workload_path, 'r') as f:
    workload = json.load(f)
print(f"Workload loaded: {workload.get('workload')}")
print(f"Number of typed questions: {len(workload.get('questions', []))}")

In [ ]:
# Step 5: Execute LoRA Fine-Tuning
# Using jaredpalmer/kev-0.8b (fast on T4) or jaredpalmer/kev-4b (higher reasoning)
BASE_MODEL = "jaredpalmer/kev-0.8b" # Change to "jaredpalmer/kev-4b" if using A100/L4

!python3 scripts/train.py \
    --init_from {BASE_MODEL} \
    --workload {workload_path} \
    --train {train_path} \
    --val {val_path} \
    --lora_r 16 \
    --lora_alpha 32 \
    --batch_size 8 \
    --learning_rate 2e-4 \
    --epochs 3 \
    --output {OUTPUT_DIR}

In [ ]:
# Step 6: Post-Training Calibration & Temperature Scaling
!python3 scripts/calibrate.py \
    --model {OUTPUT_DIR} \
    --val {val_path} \
    --output {os.path.join(OUTPUT_DIR, 'calibration.json')}

print("Training & Calibration complete!")
print(f"Trained adapter stored at: {OUTPUT_DIR}")
print("You can now download this folder to your local Mac (redagent/models/lora_adapter/) for local serving!")